In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import numpy as np
import os
import h5py
import helper_functions as hp
import data_loading as dl

GT Voltage Data Loading done!
GT Spikes Data Loading done!
SUB Voltage Data Loading done!
SUB Spikes Data Loading done!
Network configuration Loading done!
Trial Map Loading done!
Data Loading done!


In [2]:
#Behavioural Metrics
#This metric needs the output neuron


#Getting data
gt_data, gt_spikes, sub_data, sub_spikes, cfg, tmap, truth_table, h5_file_path_gt, h5_file_path_sub = dl.get_data()

#Initializing common stuff that gets repeated
spike_cols = list(set(hp.get_spike_cols(cfg, gt_data)) | set(hp.get_spike_cols(cfg, sub_data)))
patterns = tmap["case"].unique().tolist()
for patt in patterns:
    common_ids = tmap[tmap["case"] == patt]["trial_id"].tolist()

# Get output neuron spike column and trial length from metadata
role_behavioural = "output"
label = hp.get_spike_cols(cfg, gt_data, role=role_behavioural)[0]
meta = hp.load_metadata(h5_file_path_gt)
trial_len = int(meta["trial_len_ms"])

def compute_confusion_matrix(data,pattern):
    """
    For a given pattern, loops over all 10 repetitions and checks
    whether E fired within the trial window. Classifies each trial
    as TP, FN, TN or FP based on the truth table.
    """
    trials = hp.get_trials_by_pattern(data,pattern)
    TP, FN, TN, FP = 0, 0, 0, 0
    for t in trials:
        #print(t["t_in_trial"])
        window = t[t["t_in_trial"] <= trial_len]
        fired = window[label].sum() > 0
        want = truth_table[f"XOR_{pattern}"]["expected_output"]
        have = 1 if fired else 0

        if want == 1 and have == 1:
            TP += 1
        elif want == 1 and have == 0:
            FN += 1
        elif want == 0 and have == 0:
            TN += 1
        else:
            FP += 1

    return TP,FN,TN,FP
        
def all_patterns(data,patterns):
    """
    Runs compute_confusion_matrix() for all 4 XOR patterns.
    patterns derived from tmap — not hardcoded.
    Returns a DataFrame with TP/FN/TN/FP and derived metrics per pattern.
    """     
    rows = []
    for p in patterns:
        tp,fn,tn,fp = compute_confusion_matrix(data,p)
        den = tp + fn + tn + fp
        rows.append({
            "Pattern": p,
            "TP": tp, "FN": fn, "TN": tn, "FP": fp,
            "Accuracy": (tp + tn) / den if den else 0.0,
            "Sensitivity": tp / (tp + fn) if (tp + fn) else 0.0,
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        })

    return pd.DataFrame(rows)

print("Behavioural Metrics")
print(pd.DataFrame(truth_table))
gt_results = all_patterns(gt_data,patterns)
sub_results = all_patterns(sub_data,patterns)
print("Ground Truth:")
print(gt_results.to_string(index=False))
print("\nSubmission:")
print(sub_results.to_string(index=False))


GT Voltage Data Loading done!
GT Spikes Data Loading done!
SUB Voltage Data Loading done!
SUB Spikes Data Loading done!
Network configuration Loading done!
Trial Map Loading done!
Behavioural Metrics
                 XOR_00  XOR_01  XOR_10  XOR_11
input_A               0       0       1       1
input_B               0       1       0       1
expected_output       0       1       1       0
Ground Truth:
Pattern  TP  FN  TN  FP  Accuracy  Sensitivity  Specificity
     00   0   0  10   0       1.0          0.0          1.0
     11   0   0  10   0       1.0          0.0          1.0
     01  10   0   0   0       1.0          1.0          0.0
     10  10   0   0   0       1.0          1.0          0.0

Submission:
Pattern  TP  FN  TN  FP  Accuracy  Sensitivity  Specificity
     00   0   0  10   0       1.0          0.0          1.0
     11   0   0  10   0       1.0          0.0          1.0
     01   0  10   0   0       0.0          0.0          0.0
     10  10   0   0   0       1.0        